# CSV - Representation

Introduce inconsistent numeric or text representations, including nonstandard missing-value markers.

**Instructions:** Use `prepare_data_csv_update.ipynb` to create a clean `data/weather.csv` before running. This notebook overwrites it; restore the clean CSV before rerunning.

In [ ]:
from pathlib import Path
from random import Random
import pandas as pd

# Configuration
DATA_PATH = Path("data/weather.csv")
TIME_COLUMN = "time"
TIMEZONE = "America/Edmonton"
START_DATE = "2025-08-01"
END_DATE = "2025-09-01"  # Exclusive; local calendar dates.
SEED = 814205

# Representation
REPRESENTATION_TARGET_COLUMN = "relative_humidity_2m"
REPRESENTATION_KIND = "mixed"  # numeric: "numeric_suffix", "sentinel"; text: "whitespace", "case", "sentinel"; both: "mixed", "random".
REPRESENTATION_PLACEMENT = "mixed"  # "scattered", "grouped", "mixed", "random".
REPRESENTATION_RATE = (0.01, 0.03)  # Share of current eligible nonmissing values changed.
REPRESENTATION_GROUPS = 3
REPRESENTATION_GROUPED_SHARE = 0.70
NUMERIC_SUFFIX = "%"  # A plausible unit marker for the configured humidity column.
SENTINEL = "not_recorded"  # An ordinary string that read_csv does not treat as NA.


In [ ]:
# Shared placement helper: copy once into a combined dirty_data.py.
def select_positions(n, rng, *, method, rate, n_groups, grouped_share):
    """Select a fraction of the current eligible records in ordered positions.

    Grouped blocks have at least two consecutive eligible records; mixed
    placement reserves at least one independently scattered selection.
    """
    if n < 1:
        raise ValueError("No eligible records.")
    if method not in ("scattered", "grouped", "mixed", "random"):
        raise ValueError("Unknown placement method.")
    if not 0 < rate[0] <= rate[1] < 1:
        raise ValueError("Rate must be between 0 and 1.")
    if n_groups < 1 or not 0 < grouped_share < 1:
        raise ValueError("Invalid group count or grouped share.")

    chosen = rng.choice(("scattered", "grouped", "mixed")) if method == "random" else method
    count = max(1, round(n * rng.uniform(*rate)))
    if chosen == "mixed" and count < 3:
        raise ValueError("Mixed placement needs two grouped and one scattered record.")
    grouped_count = (count if chosen == "grouped" else
                     max(2, min(count - 1, round(count * grouped_share))) if chosen == "mixed" else 0)
    selected = set()
    if grouped_count:
        if grouped_count < 2:
            raise ValueError("Grouped placement requires at least two affected records.")
        groups = min(n_groups, grouped_count // 2)
        if n - grouped_count < groups - 1:
            raise ValueError("Insufficient eligible records for separate groups.")
        length, extra = divmod(grouped_count, groups)
        lengths = [length + (i < extra) for i in range(groups)]
        rng.shuffle(lengths)
        slack = n - grouped_count - (groups - 1)
        dividers = sorted(rng.sample(range(slack + groups), groups))
        gaps = ([dividers[0]] +
                [dividers[i] - dividers[i - 1] - 1 for i in range(1, groups)] +
                [slack + groups - 1 - dividers[-1]])
        cursor = gaps[0]
        for i, block_length in enumerate(lengths):
            selected.update(range(cursor, cursor + block_length))
            if i < groups - 1:
                cursor += block_length + 1 + gaps[i + 1]

    scattered_count = count - grouped_count
    if scattered_count:
        remaining = [i for i in range(n) if i not in selected]
        selected.update(rng.sample(remaining, k=scattered_count))
    if len(selected) != count:
        raise AssertionError("Selected count differs from the intended count.")
    return sorted(selected), chosen

In [ ]:
def add_representation_errors(
    df, rng, *, time_column, timezone, start_date, end_date,
    target_column, kind, placement, representation_rate, n_groups,
    grouped_share, numeric_suffix, sentinel,
):
    if time_column not in df or target_column not in df:
        raise ValueError("Time or target column not found.")
    if target_column == time_column:
        raise ValueError("Do not alter the time column with this value-representation function.")
    if not 0 < representation_rate[0] <= representation_rate[1] < 1:
        raise ValueError("Representation rate must lie between 0 and 1.")
    if placement not in ("scattered", "grouped", "mixed", "random"):
        raise ValueError("Unrecognized representation placement.")
    if not 0 < grouped_share < 1 or not numeric_suffix or not sentinel:
        raise ValueError("Provide a positive grouped share and nonempty representation markers.")
    if sentinel.lower() in ("", "na", "n/a", "null", "none", "nan"):
        raise ValueError("Use a sentinel that read_csv won't silently interpret as a null.")

    dirty = df.reset_index(drop=True).copy()
    numeric = pd.api.types.is_numeric_dtype(dirty[target_column])
    allowed = ("numeric_suffix", "sentinel") if numeric else ("whitespace", "case", "sentinel")
    if kind not in (*allowed, "mixed", "random"):
        raise ValueError(f"Kind {kind!r} is not supported for this column type.")
    selected_kind = rng.choice((*allowed, "mixed")) if kind == "random" else kind
    selected_placement = rng.choice(("scattered", "grouped", "mixed")) if placement == "random" else placement

    time = pd.to_datetime(dirty[time_column], utc=True, errors="raise")
    if time.isna().any():
        raise ValueError("Timestamps cannot be missing.")
    start = pd.Timestamp(start_date, tz=timezone).tz_convert("UTC")
    end = pd.Timestamp(end_date, tz=timezone).tz_convert("UTC")
    if start >= end:
        raise ValueError("Start date must precede end date.")

    scope = time.ge(start) & time.lt(end)
    if selected_kind in ("sentinel", "mixed") and dirty[target_column].dropna().astype(str).eq(sentinel).any():
        raise ValueError("Sentinel already exists; use a fresh source or choose a different marker.")
    eligible_mask = scope & dirty[target_column].notna()
    if not numeric and selected_kind in ("case", "mixed"):
        # Case changes require letters; numeric-looking IDs should instead use whitespace.
        eligible_mask &= dirty[target_column].astype(str).map(lambda x: any(c.isalpha() for c in x))
    eligible = dirty.loc[eligible_mask].sort_values(time_column, kind="stable")
    if eligible.empty:
        raise ValueError("No eligible values; check the column, dates, and representation kind.")

    selected, _ = select_positions(
        len(eligible), rng, method=placement, rate=representation_rate,
        n_groups=n_groups, grouped_share=grouped_share,
    )
    count = len(selected)
    if selected_kind == "mixed" and count < len(allowed):
        raise ValueError("Mixed representation needs at least one observation per style.")

    rows = eligible.index[selected]
    styles = [allowed[i % len(allowed)] for i in range(count)] if selected_kind == "mixed" else [selected_kind] * count
    rng.shuffle(styles)
    original = dirty.loc[rows, target_column].astype(str).copy()
    dirty[target_column] = dirty[target_column].astype(object)
    for row, style in zip(rows, styles):
        value = str(dirty.at[row, target_column])
        if style == "numeric_suffix":
            changed = value + numeric_suffix
        elif style == "whitespace":
            changed = value + " "
        elif style == "case":
            changed = value.upper() if value != value.upper() else value.lower()
        else:
            changed = sentinel
        dirty.at[row, target_column] = changed

    if not dirty.loc[rows, target_column].astype(str).ne(original).all():
        raise AssertionError("Some selected values did not change representation.")
    return dirty


In [ ]:
# Small trials: business-style numeric amounts and text channels.
trial_time = pd.date_range("2025-08-01", periods=120, freq="2h", tz="UTC")
orders = pd.DataFrame({
    "time": trial_time,
    "amount": [float(10 + i * 2) for i in range(120)],
    "channel": ["online", "store", "partner"] * 40,
})

trial_numeric = add_representation_errors(
    orders, Random(SEED), time_column="time", timezone="UTC",
    start_date="2025-08-01", end_date="2025-08-15",
    target_column="amount", kind="mixed", placement="mixed",
    representation_rate=(0.15, 0.15), n_groups=2,
    grouped_share=0.70, numeric_suffix=" USD", sentinel="not_recorded",
)
trial_text = add_representation_errors(
    orders, Random(SEED), time_column="time", timezone="UTC",
    start_date="2025-08-01", end_date="2025-08-15",
    target_column="channel", kind="mixed", placement="mixed",
    representation_rate=(0.15, 0.15), n_groups=2,
    grouped_share=0.70, numeric_suffix=" USD", sentinel="not_recorded",
)
assert trial_numeric["amount"].astype(str).ne(orders["amount"].astype(str)).sum() == 18
assert trial_text["channel"].ne(orders["channel"]).sum() == 18
pd.testing.assert_frame_equal(trial_numeric.drop(columns="amount"), orders.drop(columns="amount"))
pd.testing.assert_frame_equal(trial_text.drop(columns="channel"), orders.drop(columns="channel"))
trial_text.loc[trial_text["channel"].ne(orders["channel"])].head(12)

In [ ]:
# Apply to the prepared CSV and overwrite weather.csv.
clean = pd.read_csv(DATA_PATH)
original_text = pd.read_csv(DATA_PATH, dtype=str, keep_default_na=False)
dirty = add_representation_errors(
    clean, Random(SEED), time_column=TIME_COLUMN, timezone=TIMEZONE,
    start_date=START_DATE, end_date=END_DATE,
    target_column=REPRESENTATION_TARGET_COLUMN, kind=REPRESENTATION_KIND,
    placement=REPRESENTATION_PLACEMENT, representation_rate=REPRESENTATION_RATE,
    n_groups=REPRESENTATION_GROUPS, grouped_share=REPRESENTATION_GROUPED_SHARE,
    numeric_suffix=NUMERIC_SUFFIX, sentinel=SENTINEL,
)
dirty.to_csv(DATA_PATH, index=False)
saved = pd.read_csv(DATA_PATH)
saved_text = pd.read_csv(DATA_PATH, dtype=str, keep_default_na=False)

In [ ]:
# Inspect: CSV representation errors may not appear as ordinary pandas nulls.
saved.dtypes

In [ ]:
saved.isna().any().any(), saved.isna().sum()

In [ ]:
# Development-only: compare against the clean CSV to locate changed cells.
changed = saved_text[REPRESENTATION_TARGET_COLUMN].ne(original_text[REPRESENTATION_TARGET_COLUMN])
saved_text.loc[changed, [TIME_COLUMN, REPRESENTATION_TARGET_COLUMN]].head(20)

In [ ]:
# Verify the exact stored cells, including any original blank values.
assert changed.any()
assert len(saved_text) == len(original_text)
pd.testing.assert_frame_equal(
    saved_text.drop(columns=REPRESENTATION_TARGET_COLUMN), original_text.drop(columns=REPRESENTATION_TARGET_COLUMN)
)
pd.testing.assert_series_equal(
    saved_text.loc[~changed, REPRESENTATION_TARGET_COLUMN], original_text.loc[~changed, REPRESENTATION_TARGET_COLUMN]
)
pd.testing.assert_series_equal(
    saved_text.loc[changed, REPRESENTATION_TARGET_COLUMN], dirty.loc[changed, REPRESENTATION_TARGET_COLUMN].astype(str),
    check_names=False,
)
assert saved.loc[changed, REPRESENTATION_TARGET_COLUMN].notna().all()
print("Representation changes:", int(changed.sum()))
print("Sentinel values:", int(saved_text[REPRESENTATION_TARGET_COLUMN].eq(SENTINEL).sum()))
saved_text.loc[changed, [TIME_COLUMN, REPRESENTATION_TARGET_COLUMN]].head(20)

In [ ]:
import matplotlib.pyplot as plt

# Development diagnostic: affected values by local calendar date.
local_dates = pd.to_datetime(saved_text[TIME_COLUMN], utc=True).dt.tz_convert(TIMEZONE).dt.date
counts = changed.groupby(local_dates).sum()
fig, ax = plt.subplots(figsize=(10, 3.5))
ax.bar(pd.to_datetime(counts.index), counts.values, width=0.9)
ax.set(title=f"CSV representation changes: {REPRESENTATION_TARGET_COLUMN}", xlabel="Local date", ylabel="Changed values")
fig.autofmt_xdate()
plt.tight_layout()
plt.show()